<a href="https://colab.research.google.com/github/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/blob/main/_leandro_gapminder_simplificado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise do Desenvolvimento Humano e Econômico Global (Gapminder)

## 1. Título e Introdução

**Grupo:** [nomes]

Breve introdução do tema e objetivos. *(a preencher)*

**Etapas:** 1) Título e Introdução · 2) Coleta, Limpeza e AED · 3) Funções de Visualização · 4) Interpretação · 5) Síntese · 6) Conclusão

## 2. Coleta, Limpeza e Análise Exploratória (AED)

### 2.1 Coleta dos Dados

**#Importar Bibliotecas**

In [147]:
# Importar as bibliotecas que serão utilizadas pelo grupo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv
from charset_normalizer import from_bytes #Para detectar encoding e separador dos arquivos automaticamente
import urllib.request as ur   #Para aceitar base de dados que venham de URL

**#URLs das bases de Dados**

In [148]:
# Substitua a URL abaixo pela URL 'raw' do seu arquivo no GitHub
url_github =['https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/gapminder_full.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/gasto_saude_per_capita.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/medicos_por_1000_habitantes.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/mortalidade_infantil.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/taxa_obesidade.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/veiculos_por_1000_habitantes.csv']


**#Detecta o tipo de Encoding e Separador de CSVs**

In [149]:
def configuraçoes_csv(caminho_arquivo):
    """
    Descobre o encoding e o separador de qualquer CSV de forma rápida e certeira.
    Aceita tanto arquivos locais quanto links da internet (ex: GitHub).
    Retorna: (encoding, separador).
    """

    # 1. Verifica se é um link da internet ou arquivo local
    if caminho_arquivo.startswith('http://') or caminho_arquivo.startswith('https://'):   # Verifica se começa com um dos termos
        # Se for link, faz uma requisição HTTP e lê os primeiros 20.000 bytes
        req = ur.Request(
            caminho_arquivo,
            headers={'User-Agent': 'Mozilla/5.0'} # Simula um cabeçalho real na requisição como se fosse um usuário acessando pelo Mozila
        )
        with ur.urlopen(req) as f:   # Abre a conexão com o servidor através da requisição anterior criada
            amostra_binaria = f.read(20000)   #Lê e armazena os primeiros 20000 bytes em amostra_binária
                                              #NOTA: não usamos open(caminho_arquivo, "rb") pq na internet tudo já está em byte
    else:
        # Se for arquivo local, abre normalmente
        with open(caminho_arquivo, 'rb') as f:    #Aqui usamos o 'rb' pois arquivos locais não costuman estar em binário pois nós não conseguiriamos ler.
            amostra_binaria = f.read(20000)

    # 2. Descobre o encoding com quase 100% de precisão
    resultado = from_bytes(amostra_binaria).best()  # Analisa a amostra binária
    encoding = resultado.encoding   # Armazena o nome do encoding detectado
    texto_amostra = str(resultado)    # Converte em texto legível novamente

    # 3. Descobre o separador (.csv, .txt, etc)
    dialeto = csv.Sniffer().sniff(texto_amostra)    #"Fareja" em nossa amostra qual o separador mais aparece e
    separador = dialeto.delimiter

    return caminho_arquivo,encoding, separador

**#Cria um único DataFrame**

In [150]:
def cria_dataframe(caminho_arquivo,encoding,separador):
    df=pd.read_csv(caminho_arquivo, encoding=encoding, sep=separador)
    return df


**#Cria múltiplos DataFrames recebendo lista de URLs ou CSVs**

In [151]:
def carrega_multiplos_dataframes(lista_caminhos):
    """
    Recebe uma lista de URLs ou caminhos locais e retorna uma lista de DataFrames carregados
    automaticamente usando as configurações detectadas para cada um deles.

    **Exemplo de entrada:**
    lista caminhos= [url1, url2, url3, ...] ou [caminho1, caminho2, caminho3, ...]

    **Exemplo de saída:**
    dataframes= [df1, df2, df3, ...]
    """
    dataframes = []   #cria uma lista vazia para nosso append ir add oq precisa
    for caminho in lista_caminhos:
        # 1. Detecta as configurações do arquivo atual
        config = configuraçoes_csv(caminho)
        # 2. Cria o DataFrame correspondente
        df = cria_dataframe(*config)#garante que cada termo da tupla seja descompactado e utilizado na função
        # 3. Adiciona o DataFrame à lista
        dataframes.append(df)
    return dataframes

# Exemplo de uso prático:
# lista_de_fontes = [url_github]  # Você pode colocar quantos links ou caminhos quiser aqui
# meus_dfs = carregar_multiplos_dataframes(lista_de_fontes)
# meus_dfs[0].head()  # Acessa o primeiro DataFrame gerado

**#Apresenta um dicionario de nomes para cada Dataframe na lista **

In [152]:
nomes = ['gapminder', 'saude', 'medicos', 'mortalidade', 'obesidade', 'veiculos'] #Cria uma lista de nomes para nossas bases de dados
dataframe_copia = carrega_multiplos_dataframes(url_github).copy() #Copia nossa lista de tabelas para comecarmos o tratamento de dados
df_brutos = dict(zip(nomes, dataframe_copia)) #Cria Dicionários e nomeia as tabelas com os nomes da lista acima


In [153]:

print(f'{df_brutos.key()[1]}')

AttributeError: 'dict' object has no attribute 'key'

**#Função Superior para aplicar outras funções em cada um dos Dataframes de uma lista de tabelas**


In [ ]:
def aplicar_funcoes(lista_tabelas, funcao, decisor):
  """
  Aplica uma função a todas as tabelas de uma lista.

  Parâmetros:
    lista_tabelas : lista de DataFrames a processar.
    funcao        : função a ser aplicada em cada tabela.
    decisor       : 1 → transformar (guarda o resultado de volta, retorna os DataFrames com a alteracao da funcao);
                    0 → visualizar/investigar (apenas executa, não altera os Dataframes retorna informacoes,
                                                vizualizacoes ou coisa do tipo).

  Retorno:
    Lista de tabelas — transformadas (decisor=1) ou originais (decisor=0).
  """
  if decisor not in (0, 1):
    raise ValueError("decisor deve ser 1 (transformar) ou 0 (visualizar)") #Se diferente de 1 ou 0 para a execução

  if decisor == 1:
    lista_tabelas_alteradas = lista_tabelas.copy()
    for i in range(len(lista_tabelas_alteradas)):
        lista_tabelas_alteradas[i] = funcao(lista_tabelas_alteradas[i])
    return lista_tabelas_alteradas

  else:
    for tabela in lista_tabelas:
        funcao(tabela)
    return lista_tabelas

**#Faz uma cópia dos dados brutos para tratamento e limpeza**

In [ ]:
dfs_brutos = carrega_multiplos_dataframes(url_github).copy()

In [ ]:
print(f'{tabelas}')

### 2.2 Limpeza dos Dados


**#Detecta NULL,NA,e duplicados**

In [ ]:
def remove_duplicados_e_vazios(df_bruto):
    df = df_bruto.copy()

    # Conta a quantidade de LINHAS que contêm pelo menos um valor nulo/vazio ou duplicados
    qtd_linhas_null = df.isnull().any(axis=1).sum()
    qtd_linhas_vazias = df.isna().any(axis=1).sum()
    qtd_linhas_duplicadas = df.duplicated().sum()

    # Remove linhas com valores nulos/vazios ou duplicados

    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)


    print(f"""A quantidade de itens excluidos se dividem em:
              Linhas com celulas Nulas: {qtd_linhas_null}
              Linhas com celulas Vazias: {qtd_linhas_vazias}
              Linhas Duplicadas: {qtd_linhas_duplicadas}""")


**#Informa situação de linhas de cada DataFrame (NULLs,NAs e duplicados)**

In [ ]:
aplicar_funcoes(dfs_brutos, remove_duplicados_e_vazios,1)

### 2.3 Análise Exploratória (AED)
*(a fazer: mínimo 15 análises)*

## 3. Funções de Visualização e Análise Gráfica
*(a fazer: função reutilizável de gráfico)*

## 4. Interpretação das Visualizações
*(a preencher)*

## 5. Síntese dos Principais Insights
*(a preencher)*

## 6. Conclusão e Reflexão
*(a preencher)*